In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ResearchPipeline") \
    .getOrCreate()

print("Spark started")

Spark started


In [ ]:
df = spark.read.json("/content/clean_data.json")
df.show(5)

+--------------------+--------------------+--------------------+--------------------+
|             authors|           published|             summary|               title|
+--------------------+--------------------+--------------------+--------------------+
|[Cedric De Boom, ...|2023-06-07T11:08:12Z|Data science has ...|Changing Data Sou...|
|[Ian Walsh, Dmytr...|2020-06-25T12:01:39Z|Modern biology fr...|DOME: Recommendat...|
|[Felix Mohr, Jan ...|2022-01-28T14:34:32Z|Learning curves a...|Learning Curves f...|
|[Davide Cacciarel...|2023-02-17T14:24:13Z|Online active lea...|Active learning f...|
|[Maximilian P Nir...|2023-04-05T11:35:17Z|The ability to ex...|Physics-Inspired ...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn("published", to_timestamp("published"))

In [ ]:
from pyspark.sql.functions import explode

df_exploded = df.withColumn("author", explode("authors"))

In [ ]:
from pyspark.sql.functions import col

df_clean = df_exploded.dropDuplicates(["title", "author"])
df_clean = df_clean.filter(col("title").isNotNull())

In [ ]:
df_papers = df_clean.select("title", "summary", "published").dropDuplicates()

In [ ]:
df_authors = df_clean.select("author").dropDuplicates()

In [ ]:
df_paper_author = df_clean.select("title", "author").dropDuplicates()

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

df_papers = df_papers.withColumn("paper_id", monotonically_increasing_id())
df_authors = df_authors.withColumn("author_id", monotonically_increasing_id())

In [ ]:
df_paper_author_final = df_paper_author \
    .join(df_papers, "title") \
    .join(df_authors, "author") \
    .select("paper_id", "author_id")

In [ ]:
df_papers.show(5)

+--------------------+--------------------+-------------------+--------+
|               title|             summary|          published|paper_id|
+--------------------+--------------------+-------------------+--------+
|Communication-Eff...|On-device machine...|2018-11-28 10:16:18|       0|
|Machine Learning:...|Machine Learning ...|2009-11-07 02:52:53|       1|
|FeDXL: Provable F...|In this paper, we...|2022-10-26 00:23:36|       2|
|Gaming and Cooper...|The success of fe...|2025-09-02 14:55:01|       3|
|Score-based Causa...|This paper addres...|2024-02-01 18:40:03|       4|
+--------------------+--------------------+-------------------+--------+
only showing top 5 rows


In [ ]:
df_authors.show(5)

+------------+---------+
|      author|author_id|
+------------+---------+
|   Luke Benz|        0|
|  Jay Yagnik|        1|
|Iurii Kemaev|        2|
|Chelsea Finn|        3|
|    Rhoda Au|        4|
+------------+---------+
only showing top 5 rows


In [ ]:
df_paper_author_final.show(5)

+--------+---------+
|paper_id|author_id|
+--------+---------+
|       0|       89|
|       0|      803|
|       0|      510|
|       0|      186|
|       0|      720|
+--------+---------+
only showing top 5 rows


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("title")

df_papers = df_papers.withColumn("paper_id", row_number().over(window))

In [ ]:
window = Window.orderBy("author")

df_authors = df_authors.withColumn("author_id", row_number().over(window))

In [ ]:
df_paper_author_final = df_paper_author \
    .join(df_papers, "title") \
    .join(df_authors, "author") \
    .select("paper_id", "author_id")

In [ ]:
df_papers.show(5)

+--------------------+--------------------+-------------------+--------+
|               title|             summary|          published|paper_id|
+--------------------+--------------------+-------------------+--------+
|A Benchmark Study...|Machine learning ...|2023-01-26 17:49:05|       1|
|A Benchmark Study...|The proliferation...|2019-05-12 17:15:11|       2|
|A Framework for I...|The potential ben...|2018-11-26 15:35:57|       3|
|A Machine Learnin...|Significant advan...|2021-11-14 18:24:11|       4|
|A Perspective on ...|Machine learning ...|2025-02-25 09:02:02|       5|
+--------------------+--------------------+-------------------+--------+
only showing top 5 rows


In [ ]:
df_authors.show(5)

+--------------------+---------+
|              author|author_id|
+--------------------+---------+
|           Aaron Key|        1|
|         Aaron Klein|        2|
|      Abdallah Shami|        3|
|Abdennour Boulesnane|        4|
|       Abeba Birhane|        5|
+--------------------+---------+
only showing top 5 rows


In [ ]:
df_paper_author_final.show(5)

+--------+---------+
|paper_id|author_id|
+--------+---------+
|       1|      743|
|       1|      197|
|       1|      689|
|       2|       83|
|       2|      788|
+--------+---------+
only showing top 5 rows


In [ ]:
print(df_paper_author_final.count())
print(df_authors.count())
print(df_papers.count())

1025
1000
250


In [ ]:
df_papers.write.mode("overwrite").option("header", True).csv("/content/papers")
df_authors.write.mode("overwrite").option("header", True).csv("/content/authors")
df_paper_author_final.write.mode("overwrite").option("header", True).csv("/content/paper_author")